In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle
import os

In [2]:
#home captured flows
df_home_sampled = pd.read_csv("home_flows.csv").sample(n=2500, random_state=42).reset_index(drop=True)
df_home_sampled['is_beacon'] = 0

In [3]:
df_ds = pd.read_csv("UNSWNB15.csv")
df_ds.head()

C:\Users\User\AppData\Local\Temp\ipykernel_2800\2701616646.py:1: DtypeWarning: Columns (0: sport, 1: dsport, 2: ct_ftp_cmd, 3: attack_cat) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ds = pd.read_csv("UNSWNB15.csv")


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label,is_beacon
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,3,7,1,3,1,1,1,NaN,0,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,2,4,2,3,1,1,2,NaN,0,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,12,8,1,2,2,1,1,NaN,0,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,6,9,1,1,1,1,1,NaN,0,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,7,9,1,1,1,1,1,NaN,0,0


In [4]:
print(df_ds.shape,  df_ds.dtypes, df_ds.describe(), df_ds.isnull().sum(), sep = '\n')

(2540047, 50)
srcip                   str
sport                object
dstip                   str
dsport               object
proto                   str
state                   str
dur                 float64
sbytes                int64
dbytes                int64
sttl                  int64
dttl                  int64
sloss                 int64
dloss                 int64
service                 str
Sload               float64
Dload               float64
Spkts                 int64
Dpkts                 int64
swin                  int64
dwin                  int64
stcpb                 int64
dtcpb                 int64
smeansz               int64
dmeansz               int64
trans_depth           int64
res_bdy_len           int64
Sjit                float64
Djit                float64
Stime                 int64
Ltime                 int64
Sintpkt             float64
Dintpkt             float64
tcprtt              float64
synack              float64
ackdat              float64
is_sm_

In [5]:
num_conv = {
    'icmp': 1,
    'igmp': 2,
    'tcp': 6,
    'udp': 17,
    'ipip': 4,
    'ospf': 89,
    'gre': 47,
    'rsvp': 46,
    'esp': 50,
    'ah': 51,
    'sctp': 132,
    'ipv6': 41,
    'ipv6-frag': 44,
    'ipv6-route': 43,
    'ipv6-no': 59,
    'ipv6-opts': 60,
    'pim': 103,
    'rtp': 96,
    'ddp': 37,
    'ipcomp': 108,
    'ipx-n-ip': 108,
}

In [6]:
def proto_convert(value):
    if pd.isna(value):
        return 0
    s = str(value).strip().lower()
    if s.isdigit():
        return int(s)
    return num_conv.get(s, 0)

df_ds['proto'] = df_ds['proto'].apply(proto_convert)

In [7]:
print(df_ds['is_beacon'].value_counts())

is_beacon
0    2537718
1       2329
Name: count, dtype: int64


In [8]:
beacon = df_ds[df_ds['is_beacon'] == 1]
non_beacon = df_ds[df_ds['is_beacon'] == 0]
n_beacon= len(beacon)

In [9]:
non_beacon_sampled = non_beacon.sample(n=n_beacon*3, random_state = 42)

In [10]:
bal_df = pd.concat([beacon, non_beacon_sampled, df_home_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

In [11]:
bal_df['dsport'] = pd.to_numeric(bal_df['dsport'], errors='coerce').fillna(0).astype(int)

In [12]:
print("Final balanced dataset:")
print(bal_df['is_beacon'].value_counts())
print(f"Total samples: {len(bal_df)}")

Final balanced dataset:
is_beacon
0    9487
1    2329
Name: count, dtype: int64
Total samples: 11816


In [13]:
import ipaddress

bal_df['src_is_private'] = bal_df['srcip'].apply(lambda x: ipaddress.ip_address(x).is_private).astype(int)
bal_df['dst_is_private'] = bal_df['dstip'].apply(lambda x: ipaddress.ip_address(x).is_private).astype(int)
bal_df['src_is_loopback'] = bal_df['srcip'].apply(lambda x: ipaddress.ip_address(x).is_loopback).astype(int)
bal_df['dst_is_loopback'] = bal_df['dstip'].apply(lambda x: ipaddress.ip_address(x).is_loopback).astype(int)

In [14]:
bal_df['pkt_ratio']  = bal_df['Spkts']  / (bal_df['Dpkts']  + 1)
bal_df['byte_ratio'] = bal_df['sbytes'] / (bal_df['dbytes'] + 1)

In [15]:
features = [
    'dsport', 'proto', 'dur', 'smeansz', 'dmeansz', 'Sload', 'pkt_ratio', 'byte_ratio',
    'Sintpkt', 'Dintpkt', 'Sjit', 'Djit', 'tcprtt', 'is_sm_ips_ports',
    'src_is_private', 'dst_is_private', 'src_is_loopback', 'dst_is_loopback'
]

X = bal_df[features].copy()
y = bal_df['is_beacon'].copy()


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [18]:
model_rf = RandomForestClassifier(
    random_state = 42,
    n_estimators = 100,
    max_depth = 15,
    n_jobs = -1
)

model_rf.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [19]:
y_pred = model_rf.predict(X_test)

In [20]:
print(confusion_matrix(y_test, y_pred))

[[1877   21]
 [   9  457]]


In [21]:
print(classification_report(y_test, y_pred, target_names=['Non-Beacon', 'Beacon']))

              precision    recall  f1-score   support

  Non-Beacon       1.00      0.99      0.99      1898
      Beacon       0.96      0.98      0.97       466

    accuracy                           0.99      2364
   macro avg       0.98      0.98      0.98      2364
weighted avg       0.99      0.99      0.99      2364



In [22]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(model_rf, X, y, cv=5, scoring='accuracy')
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV score: {cv_scores.mean()}")
print(f"Std: {cv_scores.std()}")

Cross-validation scores: [0.98688663 0.98645789 0.98561151 0.98476513 0.98476513]
Mean CV score: 0.9856972588546065
Std: 0.0008646685595329935


In [23]:
os.makedirs("models", exist_ok=True)

with open("models/beacon_rf_model.pkl", "wb") as f:
    pickle.dump(model_rf, f)


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

scores = cross_validate(
    model_rf, X, y,
    cv=5,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
)

print(f"Accuracy:  {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}")
print(f"Precision: {scores['test_precision'].mean():.3f} ± {scores['test_precision'].std():.3f}")
print(f"Recall:    {scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}")
print(f"F1-Score:  {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")
print(f"ROC-AUC:   {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")